In [ ]:
import os
import time
import random
import pickle
import argparse

import numpy as np
import scipy.sparse as sp
import scipy.signal as sig
import scipy.stats as stats
import matplotlib.pyplot as plt
from tqdm import tqdm

## Read all the waveforms for the radar and thermal fusion

In [ ]:
# Radar
with open(f"all_radar/pred.pickle", 'rb') as f:
    data = pickle.load(f)
    radar_pred = data['pred']
    radar_gt = data['gt']
    radar_participant_task_chunk_id_list = data['participant_task_chunk_id_list']
print(len(radar_pred), len(radar_gt), len(radar_participant_task_chunk_id_list))
print()

# Thermal Camera
with open(f"all_thermal_fusion/pred.pickle", 'rb') as f:
    data = pickle.load(f)
    therm_pred = data['pred']
    therm_gt = data['gt']
    therm_participant_task_chunk_id_list = data['participant_task_chunk_id_list']
print(len(therm_pred), len(therm_gt), len(therm_participant_task_chunk_id_list))

## Concatenate the waveforms from the same trial/sample

In [ ]:
pred_waveform = {participant_task: [] for participant_task, chunk in radar_participant_task_chunk_id_list}
gt_waveform = {participant_task: [] for participant_task, chunk in radar_participant_task_chunk_id_list}
for pred, gt, participant_task_chunk_id in zip(radar_pred, radar_gt, radar_participant_task_chunk_id_list):
    participant_task = participant_task_chunk_id[0]
    chunk_id = participant_task_chunk_id[1]
    pred_waveform[participant_task].extend(pred.tolist())
    gt_waveform[participant_task].extend(gt.tolist())
for participant_task in pred_waveform:
    pred_waveform[participant_task] = np.array(pred_waveform[participant_task])
    gt_waveform[participant_task] = np.array(gt_waveform[participant_task])
radar_waveforms = {}
for participant_task in pred_waveform:
    radar_waveforms[participant_task] = {
        'pred': pred_waveform[participant_task],
        'gt': gt_waveform[participant_task]
    }
print(radar_waveforms.keys())


pred_waveform = {participant_task: [] for participant_task, chunk in therm_participant_task_chunk_id_list}
gt_waveform = {participant_task: [] for participant_task, chunk in therm_participant_task_chunk_id_list}
for pred, gt, participant_task_chunk_id in zip(therm_pred, therm_gt, therm_participant_task_chunk_id_list):
    participant_task = participant_task_chunk_id[0]
    chunk_id = participant_task_chunk_id[1]
    pred_waveform[participant_task].extend(pred.tolist())
    gt_waveform[participant_task].extend(gt.tolist())
for participant_task in pred_waveform:
    pred_waveform[participant_task] = np.array(pred_waveform[participant_task])
    gt_waveform[participant_task] = np.array(gt_waveform[participant_task])
therm_waveforms = {}
for participant_task in pred_waveform:
    therm_waveforms[participant_task] = {
        'pred': pred_waveform[participant_task],
        'gt': gt_waveform[participant_task]
    }
print(therm_waveforms.keys())

In [ ]:
len(therm_waveforms.keys()), len(radar_waveforms.keys())

# Combine both dictionaries

In [ ]:
combined_waveforms = {}
for participant_task in therm_waveforms:
    if participant_task in radar_waveforms:
        combined_waveforms[participant_task] = {
            'radar': radar_waveforms[participant_task]['pred'],
            'thermal': therm_waveforms[participant_task]['pred'],
            'gt': radar_waveforms[participant_task]['gt']
        }
        assert np.array_equal(radar_waveforms[participant_task]['gt'], therm_waveforms[participant_task]['gt']), f"GT mismatch for {participant_task}"
    else:
        print(f"Participant task {participant_task} not found in radar waveforms")

In [ ]:
combined_waveforms[participant_task]['radar'].shape, combined_waveforms[participant_task]['thermal'].shape, combined_waveforms[participant_task]['gt'].shape

In [ ]:
len(combined_waveforms)

# Chunk the data

In [ ]:
# Save as npy array. Split the waveform into 20 second chunks
fs = 15
time = 120
chunk_time = 20
root_folder = f"test_combined_respiratory_waveforms"
os.makedirs(root_folder, exist_ok=True)
for participant_task in combined_waveforms:
    save_folder = os.path.join(root_folder, participant_task)
    os.makedirs(save_folder, exist_ok=True)
    radar_waveform = combined_waveforms[participant_task]['radar']
    therm_waveform = combined_waveforms[participant_task]['thermal']
    gt_waveform = combined_waveforms[participant_task]['gt']
    num_chunks = int(np.ceil(radar_waveform.shape[0] / (fs * chunk_time)))
    print(f"Participant task {participant_task} has {num_chunks} chunks")
    radar_chunks = np.split(radar_waveform, num_chunks)
    therm_chunks = np.split(therm_waveform, num_chunks)
    gt_chunks = np.split(gt_waveform, num_chunks)
    for i in range(num_chunks):
        radar_wave = radar_chunks[i]
        therm_wave = therm_chunks[i]
        gt_wave = gt_chunks[i]
        np.save(os.path.join(save_folder, f"radar_waveform_{i}.npy"), radar_wave)
        np.save(os.path.join(save_folder, f"thermal_waveform_{i}.npy"), therm_wave)
        np.save(os.path.join(save_folder, f"gt_waveform_{i}.npy"), gt_wave)